# Auto-calibration: gauge-driven inverse fitting of resist & PSF

**The pitch.** Fab engineers spend weeks tuning resist + optics models against on-wafer CD-SEM measurements. This notebook shows how to do the same in <30 seconds on a laptop using `torch.optim.Adam` — the same optimiser everyone uses for neural networks, but pointed at *physical* parameters instead of weights.

**What you'll see.**
1. Synthetic 'measurements' from a fab whose true parameters are unknown to us.
2. A bad initial guess for `(sigma, threshold)` — the sim disagrees with the fab.
3. 200 Adam steps minimising `Σ (measured_CD − simulated_CD)²`.
4. Pre/post-fit scatter — the gap closes.

Runs on CPU, ~10–30 s end-to-end. Re-uses `simulate_aerial_image` / `differentiable_threshold` from OpenLithoHub — no new physics.

## 1. Imports & seed

In [ ]:
import math

import matplotlib.pyplot as plt
import torch
from torch.nn import functional

torch.manual_seed(0)

## 2. A small line/space gauge bank

Four target widths (8, 10, 12, 14 px) at four pitches. In a real workflow these come from a `GaugeTable` — see `openlithohub.workflow.gauges.parse_gauge`.

In [ ]:
def line_mask(width_px: int, period_px: int, size: int = 64) -> torch.Tensor:
    m = torch.zeros(size, size)
    n = size // period_px
    for k in range(n):
        col = k * period_px + (period_px - width_px) // 2
        m[:, col : col + width_px] = 1.0
    return m


GAUGES = [(8, 16), (10, 20), (12, 24), (14, 28)]  # (width_px, period_px)
masks = [line_mask(w, p) for w, p in GAUGES]
n_lines_each = [64 // p for _, p in GAUGES]
target_cds = torch.tensor([float(w) for w, _ in GAUGES])

fig, axes = plt.subplots(1, len(masks), figsize=(8, 2.2))
for ax, m, (w, p) in zip(axes, masks, GAUGES, strict=True):
    ax.imshow(m.numpy(), cmap="gray_r")
    ax.set_title(f"w={w} p={p}")
    ax.set_xticks([])
    ax.set_yticks([])
fig.suptitle("Gauge masks")
fig.tight_layout()

## 3. A gradient-friendly forward model

We want gradients to flow into `sigma`, so we build the Gaussian kernel from a *trainable* tensor instead of calling `simulate_aerial_image`'s Python-int kernel-sizing path. This is a tiny adapter — the convolution itself, the resist threshold, and the periodic-padding contract come straight from OpenLithoHub.

In [ ]:
def gaussian_kernel(sigma: torch.Tensor, radius: int) -> torch.Tensor:
    coords = torch.arange(-radius, radius + 1, dtype=torch.float32)
    g1 = torch.exp(-0.5 * (coords / sigma.clamp(min=1e-3)) ** 2)
    k = g1.unsqueeze(0) * g1.unsqueeze(1)
    return (k / k.sum()).unsqueeze(0).unsqueeze(0)


def diff_aerial(mask: torch.Tensor, sigma: torch.Tensor, radius: int = 6) -> torch.Tensor:
    k = gaussian_kernel(sigma, radius)
    inp = mask.unsqueeze(0).unsqueeze(0)
    inp = functional.pad(inp, (radius,) * 4, mode="circular")
    return functional.conv2d(inp, k).squeeze(0).squeeze(0)


def soft_cd(
    mask: torch.Tensor,
    sigma: torch.Tensor,
    thr: torch.Tensor,
    steepness: float,
    n_lines: int,
) -> torch.Tensor:
    aerial = diff_aerial(mask, sigma)
    # tensor-aware sigmoid so the gradient flows into both `sigma` and `thr`
    resist = torch.sigmoid(steepness * (aerial - thr))
    height = resist.shape[0]
    band = resist[height // 2 - 2 : height // 2 + 2]
    return band.mean(dim=0).sum() / n_lines

## 4. Synthetic 'measurements' from a hidden fab

Pretend the real fab runs at `sigma=1.7`, `threshold=0.45`. We never tell the optimiser these numbers — only the resulting CDs.

In [ ]:
TRUE_SIGMA = 1.7
TRUE_THRESHOLD = 0.45

with torch.no_grad():
    measured = torch.stack(
        [
            soft_cd(m, torch.tensor(TRUE_SIGMA), torch.tensor(TRUE_THRESHOLD), 200.0, n)
            for m, n in zip(masks, n_lines_each, strict=True)
        ]
    )
print("targets :", target_cds.tolist())
print("measured:", [round(x, 3) for x in measured.tolist()])

## 5. Bad initial guess

We start the optimiser at `sigma=2.46`, `threshold=0.62` — deliberately wrong. Compute the pre-fit error so we have a baseline.

In [ ]:
sigma_log = torch.tensor(math.log(2.46), requires_grad=True)
thr_logit = torch.tensor(0.5, requires_grad=True)  # sigmoid(0.5) ≈ 0.622

with torch.no_grad():
    s0 = torch.exp(sigma_log).item()
    t0 = torch.sigmoid(thr_logit).item()
    pre = torch.stack(
        [
            soft_cd(m, torch.tensor(s0), torch.tensor(t0), 200.0, n)
            for m, n in zip(masks, n_lines_each, strict=True)
        ]
    )
    pre_mae = (pre - measured).abs().mean().item()
print(f"initial sigma={s0:.3f}  threshold={t0:.3f}")
print(f"pre-fit MAE: {pre_mae:.3f} px")

## 6. Run Adam on the *physical* parameters

In [ ]:
opt = torch.optim.Adam([sigma_log, thr_logit], lr=0.05)
history = []
for _ in range(200):
    sigma = torch.exp(sigma_log)
    thr = torch.sigmoid(thr_logit)
    cds = torch.stack(
        [soft_cd(m, sigma, thr, 50.0, n) for m, n in zip(masks, n_lines_each, strict=True)]
    )
    loss = (cds - measured).pow(2).mean()
    opt.zero_grad()
    loss.backward()
    opt.step()
    history.append(loss.item())

fitted_sigma = float(torch.exp(sigma_log).detach())
fitted_thr = float(torch.sigmoid(thr_logit).detach())
print(f"fitted sigma={fitted_sigma:.3f}  (true {TRUE_SIGMA})")
print(f"fitted thr  ={fitted_thr:.3f}  (true {TRUE_THRESHOLD})")

## 7. Before/after — the wow panel

In [ ]:
with torch.no_grad():
    post = torch.stack(
        [
            soft_cd(m, torch.tensor(fitted_sigma), torch.tensor(fitted_thr), 200.0, n)
            for m, n in zip(masks, n_lines_each, strict=True)
        ]
    )
    post_mae = (post - measured).abs().mean().item()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 3.5))
ax1.semilogy(history)
ax1.set_xlabel("Adam step")
ax1.set_ylabel("loss (MSE)")
ax1.set_title("Calibration loss curve")
ax1.grid(True, which="both", alpha=0.3)

lo, hi = 6, 16
ax2.plot([lo, hi], [lo, hi], "k--", alpha=0.4, label="y=x")
ax2.scatter(measured.tolist(), pre.tolist(), label=f"pre-fit (MAE={pre_mae:.2f})", marker="x")
ax2.scatter(measured.tolist(), post.tolist(), label=f"post-fit (MAE={post_mae:.2f})", marker="o")
ax2.set_xlabel("measured CD (px)")
ax2.set_ylabel("simulated CD (px)")
ax2.set_title("Sim vs measurement")
ax2.set_xlim(lo, hi)
ax2.set_ylim(lo, hi)
ax2.legend()
ax2.grid(True, alpha=0.3)
fig.tight_layout()

print(f"pre-fit MAE  = {pre_mae:.3f} px")
print(f"post-fit MAE = {post_mae:.3f} px")
verdict = "calibration improved error" if post_mae < pre_mae else "WARNING: no improvement"
print(verdict)

## What you just did

You inverted on-wafer CD measurements onto resist + optics parameters using nothing but `torch.optim.Adam`. The same recipe extends to:

- **Hopkins partial-coherence parameters** (`HopkinsParams.sigma`, `na`, illumination shape) — see `openlithohub._utils.hopkins`.
- **Real gauge CSVs** — load with `parse_gauge(...)` from `openlithohub.workflow.gauges`.
- **Process-window-aware fits** — wrap the loss in `pw_fidelity_loss` from `openlithohub.workflow.process_window` to fit across dose/focus corners simultaneously.

Notes:
- The `sigma` recovery is partially degenerate with `threshold` at low steepness — this is a real lithography artefact, not a notebook bug. Add more gauges or sweep through process corners to break the degeneracy.
- The fit is differentiable end-to-end — drop a regulariser on the parameters or constrain to physical bounds with re-parametrisation (`exp` for sigma, `sigmoid` for threshold, as we did).